# FairLens SDK Tutorial

This notebook demonstrates how to use the **FairLens Python SDK** to run bias audits programmatically — right alongside your model training code.

## What you'll learn
1. Run a full dataset audit from a pandas DataFrame
2. Run an aggregate (count-based) audit without raw data
3. Generate a plain-language report
4. Download regulatory compliance reports (RBI, EU AI Act, NYC LL144)
5. Assert fairness thresholds for CI/CD pipeline integration

## Prerequisites
- FairLens backend running at `http://localhost:8000`
- `pip install requests pandas`

In [ ]:
# Install dependencies if needed
# !pip install requests pandas

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'sdk'))

from fairlens_client import FairLensClient
import pandas as pd
import json

ImportError: FairLens SDK requires 'requests'. Install it with: pip install requests

## 1. Connect to FairLens

If authentication is enabled, pass your JWT token. In dev mode (default), no token is needed.

In [ ]:
client = FairLensClient(
    "http://localhost:8000/api",
    # api_token="your-jwt-token-here",  # uncomment if auth is enabled
    poll_interval=2,
    max_wait=120,
)
print("Connected.")

## 2. Create a Synthetic Indian Lending Dataset

This simulates an NBFC loan approval dataset with realistic Indian demographic attributes:
- **Caste category** (SC, ST, OBC, General)
- **Region** (Urban, Rural)
- **Income bracket** (Low, Mid, High)
- **Gender** (Male, Female)

We intentionally inject bias: Rural applicants and SC/ST applicants get lower approval rates.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
N = 2000

df = pd.DataFrame({
    "gender":          rng.choice(["Male", "Female"], N, p=[0.55, 0.45]),
    "caste_category":  rng.choice(["SC", "ST", "OBC", "General"], N, p=[0.18, 0.08, 0.42, 0.32]),
    "region":          rng.choice(["Urban", "Rural"], N, p=[0.6, 0.4]),
    "income_bracket":  rng.choice(["Low", "Mid", "High"], N, p=[0.35, 0.45, 0.2]),
    "credit_score":    rng.normal(650, 70, N).astype(int),
})

# Inject bias into approval decisions
base_prob = 0.65
bias = np.zeros(N)
bias += np.where(df["region"] == "Rural", -0.15, 0)
bias += np.where(df["caste_category"].isin(["SC", "ST"]), -0.12, 0)
bias += np.where(df["gender"] == "Female", -0.05, 0)
bias += (df["credit_score"] - 650) / 500  # credit score influence

df["approved"] = (rng.random(N) < (base_prob + bias)).astype(int)

print(f"Dataset: {len(df)} rows")
print(f"Approval rate: {df['approved'].mean():.1%}")
print()
print("Approval by caste:")
print(df.groupby('caste_category')['approved'].mean().round(3))
print()
print("Approval by region:")
print(df.groupby('region')['approved'].mean().round(3))

## 3. Run a Full Bias Audit

One function call uploads the data, configures the audit, runs it, and waits for results.

In [ ]:
results = client.audit(
    df,
    outcome_column="approved",
    protected_attributes=["caste_category", "region", "gender"],
    favorable_outcome=1,
    domain="lending",
    org_name="Sample NBFC",
    model_name="Auto Lending v2",
)

print(f"Job ID:  {results['id']}")
print(f"Status:  {results['status']}")

## 4. Inspect Results

The results dictionary contains per-attribute fairness metrics, group stats, proxy warnings, and more.

In [ ]:
audit_data = results.get("results", {})

for attr, data in audit_data.items():
    metrics = data.get("metrics", {})
    passed = data.get("overall_passed", "?")
    dpd = metrics.get("demographic_parity_difference", {}).get("value", "n/a")
    dir_val = metrics.get("disparate_impact_ratio", {}).get("value", "n/a")
    
    status = "PASS ✅" if passed else "FAIL ❌"
    print(f"\n── {attr} ({status}) ──")
    print(f"  Demographic Parity Difference : {dpd}")
    print(f"  Disparate Impact Ratio        : {dir_val}")
    
    for group, stats in data.get("group_stats", {}).items():
        rate = stats.get('rate', 0)
        total = stats.get('total', 0)
        print(f"    {group:12s}  rate={rate:.3f}  n={total}")

## 5. Generate Plain-Language Report

In [ ]:
report = client.generate_report(results["id"])

print("=" * 60)
print("EXECUTIVE SUMMARY")
print("=" * 60)
print(report.get("executive_summary", "No summary generated."))
print()
print("PRIORITY ACTION")
print("-" * 60)
print(report.get("priority_action", "No action generated."))

## 6. Download Regulatory Compliance Reports

FairLens can generate structured compliance JSONs for:
- 🇮🇳 **RBI Digital Lending Guidelines** — for Indian NBFCs and fintechs
- 🇪🇺 **EU AI Act Article 13** — for high-risk AI systems in Europe
- 🇺🇸 **NYC Local Law 144** — for automated employment tools in NYC
- 🇺🇸 **ECOA Adverse Action** — for US credit denial notices

In [ ]:
# Download the RBI Fair Lending compliance report
rbi_report = client.regulatory_report(results["id"], "rbi_fair_lending")

print("RBI Fair Lending Assessment")
print("=" * 50)
summary = rbi_report.get("executive_summary", {})
print(f"  Overall Assessment : {summary.get('overall_fair_lending_assessment')}")
print(f"  Attributes Tested  : {summary.get('attributes_assessed')}")
print(f"  Attributes Failed  : {summary.get('attributes_failed')}")
print(f"  Proxy Risks        : {summary.get('proxy_discrimination_risks')}")
print()
print("Regulated Entity:")
for k, v in rbi_report.get("regulated_entity", {}).items():
    print(f"  {k}: {v}")

In [ ]:
# Save the RBI report to a file
with open("rbi_fair_lending_report.json", "w") as f:
    json.dump(rbi_report, f, indent=2)
print("Saved: rbi_fair_lending_report.json")

# You can also fetch other regulatory reports:
# eu_report = client.regulatory_report(results["id"], "eu_ai_act")
# nyc_report = client.regulatory_report(results["id"], "nyc_ll144")

## 7. Aggregate Audit (No Raw Data Needed)

If you only have summary statistics, you can run an aggregate audit without sharing any raw data.

In [ ]:
agg_results = client.audit_aggregate(
    attribute_name="Caste Category",
    groups=[
        {"name": "General",  "total": 640, "favorable": 500},
        {"name": "OBC",      "total": 840, "favorable": 580},
        {"name": "SC",       "total": 360, "favorable": 180},
        {"name": "ST",       "total": 160, "favorable": 65},
    ],
    domain="lending",
    org_name="Rural NBFC",
    model_name="Manual Review Process",
)

agg_data = agg_results.get("results", {})
for attr, data in agg_data.items():
    passed = data.get("overall_passed")
    status = "PASS ✅" if passed else "FAIL ❌"
    print(f"{attr}: {status}")
    for group, stats in data.get("group_stats", {}).items():
        print(f"  {group:10s}  rate={stats.get('rate', 0):.3f}  n={stats.get('total', 0)}")

## 8. CI/CD Pipeline Integration — Assert Fairness Thresholds

Use FairLens as a fairness gate in your ML pipeline. If any attribute fails, block the deployment.

In [ ]:
def fairness_gate(client, df, outcome_col, protected_attrs, favorable):
    """Run a bias audit and return True only if all attributes pass."""
    results = client.audit(
        df,
        outcome_column=outcome_col,
        protected_attributes=protected_attrs,
        favorable_outcome=favorable,
        domain="lending",
    )
    
    audit_data = results.get("results", {})
    all_pass = True
    for attr, data in audit_data.items():
        passed = data.get("overall_passed", False)
        if not passed:
            all_pass = False
            failed = [k for k, v in data.get("metrics", {}).items() if v.get("passed") is False]
            print(f"❌ FAIL: {attr} — failed metrics: {failed}")
        else:
            print(f"✅ PASS: {attr}")
    
    return all_pass


# Run the gate
passed = fairness_gate(
    client, df,
    outcome_col="approved",
    protected_attrs=["caste_category", "region"],
    favorable=1,
)

if passed:
    print("\n🟢 Fairness gate PASSED — safe to deploy.")
else:
    print("\n🔴 Fairness gate FAILED — review bias before deploying.")

## 9. Compare Audit Runs Over Time

Track fairness drift by comparing results across audit runs.

In [ ]:
# List all past audits
history = client.list_jobs()
print(f"Total audit jobs in history: {len(history)}")

# If we have at least 2 jobs, compare them
if len(history) >= 2:
    old_id = history[-2]["id"]
    new_id = history[-1]["id"]
    comparison = client.compare_jobs(old_id, new_id)
    print(f"\nComparing {old_id[:8]}... vs {new_id[:8]}...")
    print(json.dumps(comparison.get("summary", {}), indent=2))

---

## Summary

| What | How |
|------|-----|
| Full dataset audit | `client.audit(df, ...)` |
| Aggregate audit | `client.audit_aggregate(...)` |
| Plain-language report | `client.generate_report(job_id)` |
| PDF download | `client.download_report_pdf(job_id)` |
| RBI compliance | `client.regulatory_report(job_id, 'rbi_fair_lending')` |
| EU AI Act compliance | `client.regulatory_report(job_id, 'eu_ai_act')` |
| Compare runs | `client.compare_jobs(old_id, new_id)` |
| CI/CD fairness gate | Wrap `client.audit()` in an assertion |

For the full API reference, see the [Project Documentation](../docs/PROJECT_DOCUMENTATION.md).